# 26 -- The experimental chain, standardised

Re-runs every trained configuration of the chain with per-channel
standardisation of the Sentinel-1 input (training-split statistics, as in
`19`), so that all comparisons in the dissertation are like-for-like and the
pipeline matches the reference study's practice. `19` (real attributes, seed
42) already exists and is reused; `24` already evaluated it out of domain.

| tag | channels | attrs | k | data | model seed | replaces |
|---|---|---|---|---|---|---|
| `std_baseline_zeros`    | repeated | zeros | 3 | IW | 42 | 10 |
| `std_native2ch_zeros`   | native 2 | zeros | 3 | IW | 42 | 15 |
| `std_native2ch_real`    | native 2 | real  | 3 | IW | 42 | 16 |
| `std_realattrs_k7`      | repeated | real  | 7 | IW | 42 | 14 |
| `std_realattrs_seed43`  | repeated | real  | 3 | IW | 43 | seed 43 |
| `std_realattrs_seed44`  | repeated | real  | 3 | IW | 44 | third replicate |
| `std_ew_realattrs`      | repeated | real  | 3 | EW | 42 | ew/02 |
| `std_realattrs_ep200`   | repeated | real  | 3 | IW | 42 | 200 epochs -- tests the reference's budget on the leading configuration (~5 h) |

**Resumable.** Each configuration writes its own checkpoint and metrics
JSON; a configuration whose metrics file already exists is skipped, so if
the GPU session ends mid-queue, just Run All again. ~2.5 h per configuration
at 100 epochs; ~17 h for the seven.

The split is always drawn with `SPLIT_SEED = 42` (identical to `09`); only the
model initialisation seed varies. Standardisation statistics for the IW runs
are the ones `19` saved, so every IW run sees identically scaled input; the
EW run computes its own from its training split (different sensor mode,
HH+HV).

In [7]:
import os, sys, json, random, time, datetime as dt
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import rasterio

assert torch.cuda.is_available(), 'CUDA is required.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4090


In [8]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = WORKING_REPO / 'tessa_baseline'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIRS = {'IW': WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc',
           'EW': WORKING_REPO / 'input_data' / 's1_patches_tuk_ew'}
IW_STATS_PATH = OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_standardised_norm_stats.json'   # from 19
LIDAR_SURVEY_DATE = dt.date(2024, 4, 16)

TARGET_HW = (256, 256); BATCH_SIZE = 8; EPOCHS = 100; TIMESTEPS = 1000; LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15; SPLIT_SEED = 42; NOISE_SCHEDULE = 'linear'; ATTENTION_VARIANT = 'default'
BLOCK_SIZE_M, BUFFER_M = 1024.0, 150.0
NUM_WORKERS = 4
EVAL_SAMPLER_NAME = 'plms'   # 'plms' (reference study's sampler) or 'ddim'

CONFIGS = [
    dict(tag='std_baseline_zeros',   channels='repeat', attrs='zeros', k=3, data='IW', seed=42),
    dict(tag='std_native2ch_zeros',  channels='native', attrs='zeros', k=3, data='IW', seed=42),
    dict(tag='std_native2ch_real',   channels='native', attrs='real',  k=3, data='IW', seed=42),
    dict(tag='std_realattrs_k7',     channels='repeat', attrs='real',  k=7, data='IW', seed=42),
    dict(tag='std_realattrs_seed43', channels='repeat', attrs='real',  k=3, data='IW', seed=43),
    dict(tag='std_realattrs_seed44', channels='repeat', attrs='real',  k=3, data='IW', seed=44),
    dict(tag='std_ew_realattrs',     channels='repeat', attrs='real',  k=3, data='EW', seed=42),
    dict(tag='std_realattrs_ep200',  channels='repeat', attrs='real',  k=3, data='IW', seed=42, epochs=200),  # reference budget; compare with 19/24 (0.370 PLMS)
    # added 17 Sep: three more seeds of the leading configuration (tightens the seed s.d. and the single-run threshold),
    # then two more seeds of the baseline (turns real-vs-zero attributes into a 3-vs-3 comparison)
    dict(tag='std_realattrs_seed45', channels='repeat', attrs='real',  k=3, data='IW', seed=45),
    dict(tag='std_realattrs_seed46', channels='repeat', attrs='real',  k=3, data='IW', seed=46),
    dict(tag='std_realattrs_seed47', channels='repeat', attrs='real',  k=3, data='IW', seed=47),
    dict(tag='std_baseline_zeros_seed43', channels='repeat', attrs='zeros', k=3, data='IW', seed=43),
    dict(tag='std_baseline_zeros_seed44', channels='repeat', attrs='zeros', k=3, data='IW', seed=44),
]
def ckpt_path(tag): return CHECKPOINT_DIR / f's1_tuk_pcrtc_{tag}_unet_best.pth'
def metrics_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_{EVAL_SAMPLER_NAME}_validation_metrics.json'
def history_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_history.json'
def stats_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_norm_stats.json'

assert IW_STATS_PATH.exists(), f'{IW_STATS_PATH} missing -- run 19 first'
for d in S1_DIRS.values(): assert d.exists(), f'missing {d}'

In [9]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim, p_sample_loop_plms
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc
import inspect
assert 'cond_channels_per_view' in inspect.signature(ConditionalUNet.__init__).parameters, \
    'tessa_baseline ConditionalUNet lacks cond_channels_per_view -- apply the workstation patch from 15/16 first'

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

EVAL_SAMPLER = {'plms': p_sample_loop_plms, 'ddim': p_sample_loop_ddim}[EVAL_SAMPLER_NAME]
print('evaluation sampler:', EVAL_SAMPLER_NAME)

evaluation sampler: plms


## Data (identical to 09 / 19, with channel and attribute modes as switches)

In [10]:
from scipy.ndimage import median_filter

def load_sar_db(time_path, despeckle=0):
    with rasterio.open(time_path) as src:
        sar = src.read()[:2].astype(np.float32)
    sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
    if despeckle:
        sar = median_filter(sar, size=(1, despeckle, despeckle))   # per band, on linear power, at native resolution, before dB
    return 10.0 * np.log10(np.maximum(sar, 1e-12))

def build_real_attrs(s1_path, times, context_k):
    attrs_list = json.load(open(s1_path / 'attrs.json')) if (s1_path / 'attrs.json').exists() else []
    vecs = []
    for t in times:
        idx = int(t.stem[1:]); a = attrs_list[idx] if idx < len(attrs_list) else {}
        age = (dt.date.fromisoformat(a['acquisition_date']) - LIDAR_SURVEY_DATE).days / 30.0 if a.get('acquisition_date') else 0.0
        vecs.append([age, 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0, (a.get('relative_orbit_number') or 0) / 175.0, 0, 0, 0, 0, 0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()

class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k, channels, attrs_mode, norm_stats, despeckle=0):
        self.s1_dir, self.lidar_dir, self.patch_ids = Path(s1_dir), Path(lidar_dir), list(patch_ids)
        self.context_k, self.channels, self.attrs_mode, self.norm_stats, self.despeckle = context_k, channels, attrs_mode, norm_stats, despeckle
    def __len__(self): return len(self.patch_ids)
    def __getitem__(self, i):
        pid = self.patch_ids[i]
        with rasterio.open(self.lidar_dir / f'lidar_patch_{pid}.tif') as src:
            raw = src.read().astype(np.float32)
        target = raw[0]; mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        pm = float(target[mask].sum() / max(1, int(mask.sum()))); target = (target - pm) * mask
        s1_path = self.s1_dir / f's1_patch_{pid}'
        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k: raise RuntimeError(f'{s1_path} has fewer than {self.context_k} views')
        views = []
        mean, std = self.norm_stats
        for t in times:
            sar = (load_sar_db(t, self.despeckle) - mean[:, None, None]) / std[:, None, None]
            st = F.interpolate(torch.from_numpy(sar).unsqueeze(0), size=TARGET_HW, mode='bilinear', align_corners=False).squeeze(0)
            views.append(st.repeat(2, 1, 1) if self.channels == 'repeat' else st)
        cond = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k) if self.attrs_mode == 'real' else torch.zeros(8 * self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': cond.float(),
                'attrs': attrs, 'patch_mean': torch.tensor(pm), 'patch_id': pid}

def spatial_split(s1_dir):
    lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
    s1_ids = {p.name.split('_')[-1] for p in Path(s1_dir).glob('s1_patch_*') if p.is_dir()}
    paired = sorted(lidar_ids & s1_ids)
    blocks, dropped = {}, []
    for pid in paired:
        with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src: b = src.bounds
        cx, cy = (b.left + b.right) / 2, (b.bottom + b.top) / 2
        bx, by = int(cx // BLOCK_SIZE_M), int(cy // BLOCK_SIZE_M)
        d = min(cx - bx * BLOCK_SIZE_M, (bx + 1) * BLOCK_SIZE_M - cx, cy - by * BLOCK_SIZE_M, (by + 1) * BLOCK_SIZE_M - cy)
        (dropped if d < BUFFER_M else blocks.setdefault((bx, by), [])).append(pid)
    ids = list(blocks); random.Random(SPLIT_SEED).shuffle(ids)
    target_val = int(len(paired) * VAL_FRACTION); val, train, n = [], [], 0
    for bid in ids:
        if n < target_val: val.extend(blocks[bid]); n += len(blocks[bid])
        else: train.extend(blocks[bid])
    assert not (set(train) & set(val))
    return train, val, len(dropped)

def compute_stats(s1_dir, train_ids, k, despeckle=0):
    sums = np.zeros(2); sqs = np.zeros(2); count = 0
    for pid in train_ids:
        for t in sorted((Path(s1_dir) / f's1_patch_{pid}').glob('t*.tif'))[:k]:
            sar = load_sar_db(t, despeckle); sums += sar.reshape(2, -1).sum(1); sqs += (sar.reshape(2, -1) ** 2).sum(1); count += sar.shape[1] * sar.shape[2]
    mean = sums / count; std = np.sqrt(np.maximum(sqs / count - mean ** 2, 1e-12))
    return mean.astype(np.float32), std.astype(np.float32)

SPLITS = {name: spatial_split(d) for name, d in S1_DIRS.items()}
for name, (tr, va, dr) in SPLITS.items(): print(f'{name}: train={len(tr)} val={len(va)} dropped={dr}')
assert len(SPLITS['IW'][1]) == 255, 'IW split does not match 09'
iw = json.load(open(IW_STATS_PATH)); IW_STATS = (np.array(iw['mean'], np.float32), np.array(iw['std'], np.float32))
print('IW stats (from 19):', IW_STATS)

IW: train=534 val=255 dropped=887
EW: train=534 val=255 dropped=887
IW stats (from 19): (array([-18.173368, -24.003778], dtype=float32), array([2.5610027, 2.6043713], dtype=float32))


## Train + evaluate one configuration

In [11]:
def masked_mse(pred, target, mask):
    valid = mask.bool().unsqueeze(1); return ((pred - target) ** 2)[valid].mean()

def run_config(cfg):
    tag, k = cfg['tag'], cfg['k']
    epochs = cfg.get('epochs', EPOCHS)
    if metrics_path(tag).exists():
        print(f'[{tag}] metrics exist -- skipping'); return json.load(open(metrics_path(tag)))
    s1_dir = S1_DIRS[cfg['data']]; train_ids, val_ids, _ = SPLITS[cfg['data']]
    despeckle = cfg.get('despeckle', 0)
    if cfg['data'] == 'IW' and not despeckle:
        stats = IW_STATS
    else:   # EW, or filtered input: statistics from this configuration's own training split
        stats = compute_stats(s1_dir, train_ids, k, despeckle)
        json.dump({'mean': stats[0].tolist(), 'std': stats[1].tolist(), 'n_train': len(train_ids), 'k': k, 'despeckle': despeckle}, open(stats_path(tag), 'w'), indent=2)
    print(f'[{tag}] stats VV {stats[0][0]:.2f}/{stats[1][0]:.2f}  VH {stats[0][1]:.2f}/{stats[1][1]:.2f}')

    seed_everything(cfg['seed'])
    ds_tr = LidarS1Dataset(s1_dir, LIDAR_DIR, train_ids, k, cfg['channels'], cfg['attrs'], stats, despeckle)
    ds_va = LidarS1Dataset(s1_dir, LIDAR_DIR, val_ids,   k, cfg['channels'], cfg['attrs'], stats, despeckle)
    train_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    cpv = 4 if cfg['channels'] == 'repeat' else 2
    model = ConditionalUNet(in_channels=1, cond_channels=cpv * k, attr_dim=8 * k, base_channels=128, embed_dim=256, unet_depth=4,
                            attention_variant=ATTENTION_VARIANT, cond_k=k, cond_channels_per_view=cpv).to(DEVICE)
    scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE); scaler = GradScaler()

    history = {'train_loss': [], 'val_loss': []}; best = float('inf'); t0 = time.time()
    for epoch in range(epochs):
        model.train(); tr_tot = 0.0
        for b in train_loader:
            target, cond, attrs, mask = (b[key].to(DEVICE, non_blocking=True) for key in ('lidar', 's1', 'attrs', 'mask'))
            ts = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                loss = masked_mse(model(scheduler.q_sample(target, ts), cond, attrs, ts), target, mask)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); tr_tot += loss.item()
        model.eval(); va_tot = 0.0
        with torch.no_grad():
            for b in val_loader:
                target, cond, attrs, mask = (b[key].to(DEVICE, non_blocking=True) for key in ('lidar', 's1', 'attrs', 'mask'))
                ts = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
                with autocast():
                    va_tot += masked_mse(model(scheduler.q_sample(target, ts), cond, attrs, ts), target, mask).item()
        tr, va = tr_tot / max(1, len(train_loader)), va_tot / max(1, len(val_loader))
        history['train_loss'].append(tr); history['val_loss'].append(va)
        if (epoch + 1) % 10 == 0 or epoch == 0: print(f'[{tag}] epoch {epoch+1:03d}/{epochs} train={tr:.6f} val={va:.6f}  {(time.time()-t0)/60:.0f} min')
        if va < best:
            best = va
            torch.save({'model_state_dict': model.state_dict(), 'config': {**cfg, 'timesteps': TIMESTEPS, 'noise_schedule': NOISE_SCHEDULE, 'standardised': True},
                        'epoch': epoch + 1, 'val_loss': va}, ckpt_path(tag))
    json.dump(history, open(history_path(tag), 'w'))

    # evaluate best checkpoint with EVAL_SAMPLER (PLMS, the reference study's choice; 24 showed +0.05..0.07 over DDIM)
    model.load_state_dict(torch.load(ckpt_path(tag), map_location=DEVICE)['model_state_dict']); model.eval()
    seed_everything(SPLIT_SEED); rows = []
    with torch.no_grad():
        for b in val_loader:
            target, cond, attrs = b['lidar'].to(DEVICE), b['s1'].to(DEVICE), b['attrs'].to(DEVICE)
            mask = b['mask'].to(DEVICE).bool(); pm = b['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
            pred = EVAL_SAMPLER(model, scheduler, target.shape, cond, attrs, DEVICE)
            gt_abs, pred_abs = target + pm, pred + pm
            for i, pid in enumerate(b['patch_id']):
                g, p, m = gt_abs[i], pred_abs[i], mask[i]
                gv, pv = g.squeeze()[m].cpu().numpy(), p.squeeze()[m].cpu().numpy()
                rows.append({'patch_id': pid, 'rmse_m': float(rmse(g, p, m)), 'bias_m': float(bias(g, p, m)),
                             'sigma_error_pct': float(sigma_error(g, p, m)),
                             'normal_angle_error_deg': float(normal_angle_error(g, p, m, pixel_size=1.0, degrees=True)),
                             'jsd': float(average_jsd_multiscale(g, p, pixel_size=1.0, mask=m)),
                             'psd_rmse': float(log_psd_rmse(g, p, pixel_size=1.0, mask=m)), 'zncc': float(zncc(g, p, m)),
                             'gt_std_val': float(gv.std()), 'pred_std_val': float(pv.std())})
    json.dump(rows, open(metrics_path(tag), 'w'), indent=2)
    mean = {key: float(np.nanmean([r[key] for r in rows])) for key in rows[0] if key != 'patch_id'}
    print(f'[{tag}] DONE  ZNCC {mean["zncc"]:+.4f}  RMSE {mean["rmse_m"]:.4f}  sig% {mean["sigma_error_pct"]:.1f}  pred/gt {mean["pred_std_val"]:.4f}/{mean["gt_std_val"]:.4f}  ({(time.time()-t0)/3600:.1f} h)')
    del model, optimizer; torch.cuda.empty_cache()
    return rows

## Run the queue (re-run this cell after any interruption; finished configurations are skipped)

In [12]:
results = {}
for cfg in CONFIGS:
    try:
        results[cfg['tag']] = run_config(cfg)
    except Exception as exc:
        print(f'[{cfg["tag"]}] FAILED: {type(exc).__name__}: {exc}')
        raise

[std_baseline_zeros] metrics exist -- skipping
[std_native2ch_zeros] metrics exist -- skipping
[std_native2ch_real] metrics exist -- skipping
[std_realattrs_k7] metrics exist -- skipping
[std_realattrs_seed43] metrics exist -- skipping
[std_realattrs_seed44] metrics exist -- skipping
[std_ew_realattrs] metrics exist -- skipping
[std_realattrs_ep200] stats VV -18.17/2.56  VH -24.00/2.60


/tmp/ipykernel_113079/3383517186.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE); scaler = GradScaler()
/tmp/ipykernel_113079/3383517186.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_113079/3383517186.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[std_realattrs_ep200] epoch 001/200 train=0.047304 val=0.023424  0 min
[std_realattrs_ep200] epoch 010/200 train=0.013630 val=0.015260  3 min
[std_realattrs_ep200] epoch 020/200 train=0.012752 val=0.015459  5 min
[std_realattrs_ep200] epoch 030/200 train=0.013313 val=0.014829  7 min
[std_realattrs_ep200] epoch 040/200 train=0.013540 val=0.015273  10 min
[std_realattrs_ep200] epoch 050/200 train=0.012627 val=0.014616  12 min
[std_realattrs_ep200] epoch 060/200 train=0.012560 val=0.015374  14 min
[std_realattrs_ep200] epoch 070/200 train=0.012919 val=0.014509  17 min
[std_realattrs_ep200] epoch 080/200 train=0.012374 val=0.014578  19 min
[std_realattrs_ep200] epoch 090/200 train=0.012392 val=0.014909  21 min
[std_realattrs_ep200] epoch 100/200 train=0.011904 val=0.015942  23 min
[std_realattrs_ep200] epoch 110/200 train=0.009916 val=0.015373  26 min
[std_realattrs_ep200] epoch 120/200 train=0.006870 val=0.017674  28 min
[std_realattrs_ep200] epoch 130/200 train=0.003447 val=0.023117  30 

## Summary against the unstandardised chain

In [13]:
PAIRS = [  # (standardised tag, unstandardised metrics file, label)
    ('std_baseline_zeros',   's1_pcrtc_baseline_spatialsplit_validation_metrics.json',           'repeated ch., zero attrs'),
    (None,                   's1_pcrtc_realattrs_spatialsplit_validation_metrics.json',          'repeated ch., real attrs (seed 42)'),
    ('std_native2ch_zeros',  's1_pcrtc_native2ch_spatialsplit_validation_metrics.json',          'native 2 ch., zero attrs'),
    ('std_native2ch_real',   's1_pcrtc_native2ch_realattrs_spatialsplit_validation_metrics.json','native 2 ch., real attrs'),
    ('std_realattrs_k7',     's1_pcrtc_realattrs_spatialsplit_k7_validation_metrics.json',       'k = 7'),
    ('std_realattrs_seed43', 's1_pcrtc_realattrs_spatialsplit_seed43_validation_metrics.json',   'seed 43'),
    ('std_realattrs_seed44', 's1_pcrtc_dem_realattrs_spatialsplit_validation_metrics.json',      'third replicate'),
    ('std_ew_realattrs',     's1_ew_realattrs_spatialsplit_validation_metrics.json',             'EW 40 m'),
    ('std_realattrs_ep200',  's1_pcrtc_standardised_spatialsplit_plms_validation_metrics.json',  '200 epochs (vs 19 @100, PLMS)'),
]
STD19 = OUTPUT_DIR / ('s1_pcrtc_standardised_spatialsplit_plms_validation_metrics.json' if EVAL_SAMPLER_NAME == 'plms' else 's1_pcrtc_realattrs_spatialsplit_standardised_validation_metrics.json')  # 19's run, PLMS re-evaluated by 24
def mean_of(path):
    if not Path(path).exists(): return None
    rows = json.load(open(path)); return {k: float(np.nanmean([r[k] for r in rows])) for k in rows[0] if k != 'patch_id'}
print(f'NOTE: unstd columns are DDIM (Stage 1); std columns are {EVAL_SAMPLER_NAME.upper()}. Sampler effect is in 24.')
print(f'{"configuration":<36}{"unstd ZNCC":>11}{"std ZNCC":>10}{"delta":>8}   {"unstd sig%":>10}{"std sig%":>9}   {"unstd pred":>10}{"std pred":>9}')
print('-' * 110)
for tag, unstd_file, label in PAIRS:
    u = mean_of(OUTPUT_DIR / unstd_file); s = mean_of(STD19 if tag is None else metrics_path(tag))
    f = lambda m, k, w: (f'{m[k]:>{w}.4f}' if m else f'{"--":>{w}}')
    d = f'{s["zncc"]-u["zncc"]:>+8.4f}' if (u and s) else f'{"--":>8}'
    print(f'{label:<36}{f(u,"zncc",11)}{f(s,"zncc",10)}{d}   {f(u,"sigma_error_pct",10)}{f(s,"sigma_error_pct",9)}   {f(u,"pred_std_val",10)}{f(s,"pred_std_val",9)}')
# seed statistics for the standardised real-attribute configuration
z = [m['zncc'] for m in (mean_of(STD19), mean_of(metrics_path('std_realattrs_seed43')), mean_of(metrics_path('std_realattrs_seed44'))) if m]
if len(z) == 3:
    z = np.array(z); sd = z.std(ddof=1); sd_diff = sd * np.sqrt(1 + 1/3)
    print(f'\nstandardised real-attribute, three seeds: {z.round(4).tolist()}  mean {z.mean():.4f}  sd {sd:.4f}  threshold t(2) {4.303*sd_diff:.4f}  permissive {2*sd_diff:.4f}')
# --- 17 Sep additions: all real-attribute seeds, and baseline seeds ---
def z_of(tag):
    m = mean_of(STD19 if tag is None else metrics_path(tag)); return None if m is None else m['zncc']
real = [v for v in [z_of(None)] + [z_of(f'std_realattrs_seed{s}') for s in (43, 44, 45, 46, 47)] if v is not None]
base = [v for v in [z_of('std_baseline_zeros')] + [z_of(f'std_baseline_zeros_seed{s}') for s in (43, 44)] if v is not None]
from scipy import stats as sps
if len(real) >= 3:
    n = len(real); sd = np.std(real, ddof=1); unit = sd * np.sqrt(1 + 1 / n); t = sps.t.ppf(0.975, n - 1)
    print(f'\nreal-attribute seeds (n={n}): {[round(v,4) for v in real]}  mean {np.mean(real):.4f}  sd {sd:.4f}  '
          f'single-run threshold t({n-1})={t:.3f} x {unit:.4f} = {t*unit:.4f}  permissive {2*unit:.4f}')
if len(base) >= 2:
    print(f'baseline seeds (n={len(base)}): {[round(v,4) for v in base]}  mean {np.mean(base):.4f}  sd {np.std(base, ddof=1):.4f}')
if len(real) >= 3 and len(base) >= 3:
    tt = sps.ttest_ind(real, base, equal_var=False)
    print(f'real vs baseline (Welch): diff {np.mean(real)-np.mean(base):+.4f}  t={tt.statistic:.2f}  p={tt.pvalue:.3f}')


NOTE: unstd columns are DDIM (Stage 1); std columns are PLMS. Sampler effect is in 24.
configuration                        unstd ZNCC  std ZNCC   delta   unstd sig% std sig%   unstd pred std pred
--------------------------------------------------------------------------------------------------------------
repeated ch., zero attrs                 0.1867    0.3142 +0.1274      30.4628  20.9724       0.1971   0.1404
repeated ch., real attrs (seed 42)       0.2344    0.3700 +0.1356      22.7296  37.8081       0.1384   0.1056
native 2 ch., zero attrs                 0.2068    0.3598 +0.1530      23.9357  34.6684       0.1798   0.1064
native 2 ch., real attrs                 0.2012    0.3690 +0.1677      20.6425  32.1855       0.1584   0.1114
k = 7                                    0.2878    0.3740 +0.0862      55.4569  25.0736       0.2486   0.1281
seed 43                                  0.2645    0.3558 +0.0913      21.0375  30.3550       0.1699   0.2024
third replicate                 